In [1]:
# Part 0: Imports and environment
from dotenv import load_dotenv
import os
import json
import re
from typing import Any
from pathlib import Path

import pandas as pd
import boto3

load_dotenv(override=True)

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION", "ap-south-1")

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise RuntimeError("AWS credentials are missing. Check your .env file.")

print("Environment loaded.")
print(f"Region: {AWS_DEFAULT_REGION}")

Environment loaded.
Region: ap-south-1


In [2]:
# Part 1: Create clients and fetch Bedrock catalog
def create_bedrock_session_and_clients():
    """Create AWS session plus Bedrock management/runtime clients."""
    sess = boto3.Session(
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        aws_session_token=AWS_SESSION_TOKEN,
        region_name=AWS_DEFAULT_REGION,
    )
    reg = sess.region_name or AWS_DEFAULT_REGION or "ap-south-1"
    mgmt = sess.client("bedrock", region_name=reg)
    runtime = sess.client("bedrock-runtime", region_name=reg)
    return sess, reg, mgmt, runtime

def fetch_all_foundation_models(client):
    models = []
    next_token = None
    while True:
        kwargs = {"byOutputModality": "TEXT"}
        if next_token:
            kwargs["nextToken"] = next_token
        resp = client.list_foundation_models(**kwargs)
        models.extend(resp.get("modelSummaries", []))
        next_token = resp.get("nextToken")
        if not next_token:
            break
    return models

def fetch_all_inference_profiles(client):
    profiles = []
    next_token = None
    while True:
        kwargs = {"typeEquals": "SYSTEM_DEFINED"}
        if next_token:
            kwargs["nextToken"] = next_token
        resp = client.list_inference_profiles(**kwargs)
        profiles.extend(resp.get("inferenceProfileSummaries", []))
        next_token = resp.get("nextToken")
        if not next_token:
            break
    return profiles

session, region, bedrock, bedrock_runtime = create_bedrock_session_and_clients()
foundation_models = fetch_all_foundation_models(bedrock)
inference_profiles = fetch_all_inference_profiles(bedrock)

print("Bedrock clients initialized successfully")
print(f"Region: {region}")
print(f"Total text models in catalog: {len(foundation_models)}")

Bedrock clients initialized successfully
Region: ap-south-1
Total text models in catalog: 61


In [3]:
# Part 2: Lock to Qwen3 VL 235B A22B
from botocore.exceptions import ClientError

TARGET_QWEN_MODEL_ID = "qwen.qwen3-vl-235b-a22b"
print(f"Target model ID: {TARGET_QWEN_MODEL_ID}")

matched_target_models = [
    m for m in foundation_models
    if str(m.get("modelId", "")).strip().lower() == TARGET_QWEN_MODEL_ID.lower()
]

if not matched_target_models:
    raise RuntimeError(
        f"Target model not found in this region/account catalog: {TARGET_QWEN_MODEL_ID}"
    )

chosen_model_id = TARGET_QWEN_MODEL_ID
model_name = matched_target_models[0].get("modelName", "")
print("Status: FOUND")
print(f"Model name: {model_name}")
print(f"chosen_model_id: {chosen_model_id}")

Target model ID: qwen.qwen3-vl-235b-a22b
Status: FOUND
Model name: Qwen3 VL 235B A22B
chosen_model_id: qwen.qwen3-vl-235b-a22b


In [4]:
# Part 3: Pick one image and define extraction prompt
image_root = Path(r"C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\Document_3")
image_exts = {".jpg", ".jpeg", ".png", ".webp"}
image_files = sorted([p for p in image_root.rglob("*") if p.suffix.lower() in image_exts])

if not image_files:
    raise RuntimeError(f"No image files found under {image_root}")

# Avoid likely cover/blank pages by preferring the first non-page_1 file when available.
non_cover = [p for p in image_files if "page_1" not in p.stem.lower()]
image_path = non_cover[0] if non_cover else image_files[0]

ext = image_path.suffix.lower().lstrip(".")
image_format = "jpeg" if ext == "jpg" else ext
if image_format not in {"jpeg", "png", "webp", "gif"}:
    image_format = "jpeg"

with open(image_path, "rb") as f:
    image_bytes = f.read()

print(f"Using image: {image_path}")
print(f"Image format for Bedrock: {image_format}")
print(f"Image bytes: {len(image_bytes)}")

qwen_extraction_prompt = """
You are a medical document OCR extractor.

Read only the provided image and return ONLY valid JSON.

Rules:
1. Extract every visible value.
2. If a field is not visible, keep it empty.
3. Do not fabricate data.
4. For every extracted field, provide an approximate bounding box.
5. Bounding box format:
   [x_min, y_min, x_max, y_max]
6. Coordinates should be normalized between 0 and 1000.
7. If location cannot be determined, return [].

Schema:
{
  "claimant_name": {
    "value": "",
    "bbox": []
  },
  "claimant_number": {
    "value": "",
    "bbox": []
  },
  "tax_id": {
    "value": "",
    "bbox": []
  },
  "practice_address": {
    "value": "",
    "bbox": []
  },
  "billing_address": {
    "value": "",
    "bbox": []
  },
  "diagnosis_codes": [],
  "date_of_service": {
    "value": "",
    "bbox": []
  },
  "invoice_date": {
    "value": "",
    "bbox": []
  },
  "invoice_number": {
    "value": "",
    "bbox": []
  },
  "taxonomy": {
    "value": "",
    "bbox": []
  },
  "total_amount": {
    "value": "",
    "bbox": []
  }
}
"""

print("Prompt ready.")

Using image: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\Document_3\page_1.jpg
Image format for Bedrock: jpeg
Image bytes: 289029
Prompt ready.


In [5]:
# Part 4: Invoke selected Qwen model and parse response
if not chosen_model_id:
    raise RuntimeError("chosen_model_id is not set. Run Part 2 first.")

if chosen_model_id != "qwen.qwen3-vl-235b-a22b":
    raise RuntimeError(
        f"This notebook is locked to qwen.qwen3-vl-235b-a22b, got: {chosen_model_id}"
    )

bedrock_runtime = session.client("bedrock-runtime", region_name=region)
used_qwen_model_id = None
qwen_raw_response = None
qwen_parsed = None

def try_parse_json_payload(raw_text: str) -> Any:
    if not raw_text:
        return None

    try:
        return json.loads(raw_text)
    except Exception:
        pass

    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start != -1 and end != -1 and end > start:
        chunk = raw_text[start:end + 1]
        try:
            return json.loads(chunk)
        except Exception:
            return None

    return None

try:
    resp = bedrock_runtime.converse(
        modelId=chosen_model_id,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": image_format,
                            "source": {"bytes": image_bytes},
                        }
                    },
                    {"text": qwen_extraction_prompt},
                ],
            }
        ],
        inferenceConfig={"maxTokens": 1500, "temperature": 0},
    )

    parts = resp.get("output", {}).get("message", {}).get("content", [])
    text_parts = [
        part.get("text", "")
        for part in parts
        if isinstance(part, dict) and "text" in part
    ]
    qwen_raw_response = "\n".join(t for t in text_parts if t).strip()
    used_qwen_model_id = chosen_model_id
    qwen_parsed = try_parse_json_payload(qwen_raw_response)

except ClientError as e:
    code = e.response.get("Error", {}).get("Code", "Unknown")
    msg = e.response.get("Error", {}).get("Message", str(e))
    raise RuntimeError(f"{chosen_model_id} -> {code}: {msg}") from e
except Exception as e:
    raise RuntimeError(f"{chosen_model_id} -> {str(e)}") from e

print(f"Used Qwen model: {used_qwen_model_id}")
print("Raw response preview:")
print((qwen_raw_response or "")[:1200])

if qwen_parsed is not None:
    print("\nParsed JSON:")
    print(json.dumps(qwen_parsed, indent=2, ensure_ascii=False))
else:
    print("\nCould not parse strict JSON; keeping raw response in qwen_raw_response.")

Used Qwen model: qwen.qwen3-vl-235b-a22b
Raw response preview:
{
  "claimant_name": {
    "value": "Jason Walker",
    "bbox": [57, 193, 152, 207]
  },
  "claimant_number": {
    "value": "FL197764",
    "bbox": [720, 287, 789, 301]
  },
  "tax_id": {
    "value": "85-1557257",
    "bbox": [67, 43, 187, 56]
  },
  "practice_address": {
    "value": "P.O. Box 48, Hemlock, MI 48626-0048 US",
    "bbox": [67, 62, 271, 97]
  },
  "billing_address": {
    "value": "Jason Walker, AAA Insurance, P.O Box 2946, Clinton, IA 52733-2946",
    "bbox": [57, 175, 220, 267]
  },
  "diagnosis_codes": [
    "ICD 814.109S"
  ],
  "date_of_service": {
    "value": "10/06/2025",
    "bbox": [57, 372, 135, 385]
  },
  "invoice_date": {
    "value": "10/08/2025",
    "bbox": [534, 221, 611, 235]
  },
  "invoice_number": {
    "value": "251006-3",
    "bbox": [788, 28, 893, 46]
  },
  "taxonomy": {
    "value": "HCCPS E1399",
    "bbox": [185, 387, 290, 400]
  },
  "total_amount": {
    "value": "$650.00",
  

In [6]:
# Part 5: Save model extraction result
output_dir = Path(r"C:\Users\Rohit.Sahu\Desktop\experiment\extraction_results")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "qwen_230b_single_image_extraction.json"
result_to_save = qwen_parsed if qwen_parsed is not None else {"raw_response": qwen_raw_response}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result_to_save, f, indent=2, ensure_ascii=False)

print(f"Saved extraction output to: {output_file}")

Saved extraction output to: C:\Users\Rohit.Sahu\Desktop\experiment\extraction_results\qwen_230b_single_image_extraction.json


In [7]:
from PIL import Image, ImageDraw

image_path=r'C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\Document_3\page_1.jpg'
img = Image.open(image_path)
draw = ImageDraw.Draw(img)

img_w, img_h = img.size

for field, data in qwen_parsed.items():

    if not isinstance(data, dict):
        continue

    bbox = data.get("bbox", [])

    if len(bbox) != 4:
        continue

    x1 = bbox[0] * img_w / 1000
    y1 = bbox[1] * img_h / 1000
    x2 = bbox[2] * img_w / 1000
    y2 = bbox[3] * img_h / 1000

    draw.rectangle(
        [x1, y1, x2, y2],
        outline="red",
        width=4
    )

    draw.text(
        (x1, max(0, y1 - 20)),
        field,
        fill="red"
    )

img.save("highlighted_image.png")
img.show()